# MC-DCNN over the full TS - Final run
- Use ROC-AUC as metric.
- Add kernel initializers
- Reduce regularization to allow the model to learn more
- More aggressive Early Stopping callback
- Reduced Search Space

In [ ]:
import os
# Make sure XLA is OFF (Metal doesn't support XLA/JIT)
# os.environ.pop("TF_XLA_FLAGS", None)
# os.environ.pop("XLA_FLAGS", None)
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"


import tensorflow as tf
# tf.config.optimizer.set_jit(False)
print("GPUs:", tf.config.list_physical_devices("GPU"))


import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    average_precision_score,
    PrecisionRecallDisplay,
    accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score
)

import kineticstoolkit as ktk

import random
import pickle
import json
from typing import Tuple

from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras import layers as L, models as M, Input, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy

from core.matlab_data_loader import MatlabDataLoader
from core.processing import preprocess_features
from core.timeseries import plot_compare_features, plot_all_features_overlay, bilateral_to_unilateral, split_unilateral
from core.evaluation import model_test_summary, BilateralSingleInputPredictor, pick_threshold
from core.tunning import MetaHyperModel, ModelLoader, summarize_best_N_models


from pathlib import Path
import core.constants as c

RANDOM_STATE = 42
RANDOM_STATE_2 = 6
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Optimization for M4
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

## Notebook configuration

- **SAVE_MODEL**  
  - `True`: Will run and save this iteration of the model.  
  - `False`: Will load the model from the results folder.

- **SAVE_DATA**  
  - `True`: Will save the train, test, and validation sets as npz files.  
  - `False`: Will load the data from the results folder.

- **USE_TENSORBOARD**  
  - `True`: Will use TensorBoard to visualize the training process.
  - `False`: Will not use TensorBoard.

- **MODEL_NAME**  
  - The name of the model.

- **MODEL_RESULTS_FOLDER**  
  - The folder where the model results will be saved.

In [ ]:
MODEL_NAME = "mc-dcnn-v4-final"
MODEL_RESULTS_FOLDER = Path(c.RICKD_MODELS_FOLDER)/ "deep_learning" / MODEL_NAME
MODEL_RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Model results folder: {MODEL_RESULTS_FOLDER}")

In [ ]:
SAVE_MODEL = True
SAVE_DATA = False
USE_TENSORBOARD = False

print(f"SAVE_MODEL: {SAVE_MODEL}")
print(f"SAVE_DATA: {SAVE_DATA}")
print(f"USE_TENSORBOARD: {USE_TENSORBOARD}")

## Load Input Data

In [ ]:
timeseries_folder = Path(c.RICKD_PROCESSED_DATA_FOLDER) / 'timeseries'

all_sessions_matrix = np.load(timeseries_folder / 'timeseries_mean_matrix.npy')
with open(timeseries_folder / 'timeseries_mean_channels.json', 'r') as f:
    channels = json.load(f)
with open(timeseries_folder / 'timeseries_mean_sessions.json', 'r') as f:
    valid_session_ids = json.load(f)

# Edge case for subject created during DQ:
valid_session_ids = [
    '300375_20140502T074159' if sid == '200375_20140502T074159' else sid
    for sid in valid_session_ids
]

print("Shape of the timeseries matrix: ", all_sessions_matrix.shape)
print("Number of channels: ", len(channels))
print("Number of sessions: ", len(valid_session_ids))

for channel in channels:
    print(channel)

In [ ]:
loader = MatlabDataLoader()

session_data = loader.get_session_data_full_cleaned().set_index("id")
metadata_columns = ["is_injured", "sub_id"]
# Filter session_data to only include valid_session_ids and preserve their order
session_data = session_data.loc[session_data.index.intersection(valid_session_ids)]
session_data = session_data.reindex(valid_session_ids)
session_data

In [ ]:
from core.utils import extract_subject_id

# N = 1456 -- Sessions with complete metadata and present in time-series.
X_ts: np.ndarray = all_sessions_matrix.astype(np.float32)  #  (N, 101, 54)
y: pd.Series = np.array(session_data["is_injured"].values, dtype=np.float32)  # (N,)
subject_id: pd.Series = extract_subject_id(session_data.index)  # (N,)

## Preprocessing of input data
- Scale with Z-Score & one-hot encode categorical cols
- Split data into train, validation and test sets

In [ ]:
print("="*50)
print("Data overview")
print("="*50)
print(f"Timeseries shape: {X_ts.shape}")  # (N, 101, 48)
print(f"Labels shape: {y.shape}")        # (N,)
print(f"Subject IDs shape: {subject_id.shape}")  # (N,)

# Check class distribution
unique, counts = np.unique(y, return_counts=True)
print(f"\nClass distribution:")
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} samples ({count/len(y)*100:.1f}%)")

# Check for any missing values
print(f"\nMissing values in timeseries: {np.isnan(X_ts).sum()}")

In [ ]:
from core.evaluation import standardise_and_split_ts, verify_splits

print("Train-Test-Val Split (Group-aware by subject_id)")
print("="*50)
data, scaler_ts = standardise_and_split_ts( X_ts, y, subject_id,
    test_size=0.2,
    val_size=0.2,
    random_state=42,
)

(
    X_ts_train, X_ts_val, X_ts_test,
    y_train, y_val, y_test,
    subject_train, subject_val, subject_test,
) = data

print("Shapes:")
for name, arr in [
    ("X_train", X_ts_train), ("y_train", y_train),
    ("X_val", X_ts_val), ("y_val", y_val),
    ("X_test", X_ts_test), ("y_test", y_test),
]:
    shape = arr.shape if hasattr(arr, "shape") else (len(arr),)
    print(f"{name}: {shape}")

verify_splits(X_ts, X_ts_train, X_ts_val, X_ts_test, y_train, y_val, y_test, subject_train, subject_val, subject_test)

In [ ]:
# To handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print(f"Class weights")
print("="*50)
print(f"Class weights: {class_weight_dict}")

In [ ]:
# Save train, test, and validation sets as npz files (with inverted channels)
if SAVE_DATA:
    print("Saving data to:")
    print("  ", MODEL_RESULTS_FOLDER / "train.npz")
    print("  ", MODEL_RESULTS_FOLDER / "val.npz")
    print("  ", MODEL_RESULTS_FOLDER / "test.npz")
    print()

    np.savez_compressed(
        MODEL_RESULTS_FOLDER / "train.npz",
        X_ts=X_ts_train,
        y=y_train,
        subject=subject_train
    )
    np.savez_compressed(
        MODEL_RESULTS_FOLDER / "val.npz",
        X_ts=X_ts_val,
        y=y_val,
        subject=subject_val
    )
    np.savez_compressed(
        MODEL_RESULTS_FOLDER / "test.npz",
        X_ts=X_ts_test,
        y=y_test,
        subject=subject_test
    )

In [ ]:
if not SAVE_DATA:
    print("Loading data from:")
    print("  ", MODEL_RESULTS_FOLDER / "train.npz")
    print("  ", MODEL_RESULTS_FOLDER / "val.npz")
    print("  ", MODEL_RESULTS_FOLDER / "test.npz")
    print()

    # Set allow_pickle=True to allow loading object arrays (e.g., for subject arrays)
    train_data = np.load(MODEL_RESULTS_FOLDER / "train.npz", allow_pickle=True)
    val_data   = np.load(MODEL_RESULTS_FOLDER / "val.npz", allow_pickle=True)
    test_data  = np.load(MODEL_RESULTS_FOLDER / "test.npz", allow_pickle=True)

    X_ts_train = train_data["X_ts"]
    y_train = train_data["y"]
    subject_train = train_data["subject"]

    X_ts_val = val_data["X_ts"]
    y_val = val_data["y"]
    subject_val = val_data["subject"]

    X_ts_test = test_data["X_ts"]
    y_test = test_data["y"]
    subject_test = test_data["subject"]

print("Train set shapes:")
print("  X_ts_train:", X_ts_train.shape)
print("  y_train:", y_train.shape)
print("  subject_train:", subject_train.shape)
print()
print("Validation set shapes:")
print("  X_ts_val:", X_ts_val.shape)
print("  y_val:", y_val.shape)
print("  subject_val:", subject_val.shape)
print()
print("Test set shapes:")
print("  X_ts_test:", X_ts_test.shape)
print("  y_test:", y_test.shape)
print("  subject_test:", subject_test.shape)
print()

def class_balance_percent(y):
    counts = np.bincount(y.astype(int))
    total = counts.sum()
    return [f"{100 * c / total:.1f}%" for c in counts]

print("Class balance (train):", class_balance_percent(y_train))
print("Class balance (val):", class_balance_percent(y_val))
print("Class balance (test):", class_balance_percent(y_test))

## Building the model

In [ ]:
@tf.keras.utils.register_keras_serializable(package="custom")
class ChannelSlice(L.Layer):
    """x[:, :, index:index+1] with proper serialization."""
    def __init__(self, index: int, **kwargs):
        super().__init__(**kwargs)
        self.index = int(index)
    def call(self, x):
        return x[:, :, self.index:self.index+1]
    def get_config(self):
        return {"index": self.index}

def _kernel_init_for(act: str):
    # ReLU-like -> He; Sigmoid/Tanh -> Glorot
    return (tf.keras.initializers.HeNormal()
            if act.lower() in ("relu")
            else tf.keras.initializers.GlorotUniform())

def build_model(hp: kt.HyperParameters,
                model_name: str,
                features: int = 48,
                time_steps: int = 101,
                clipnorm: float = 0.1):
    # Hyperparameters
    # Architecture
    activation_combo = hp.Choice(
        # "activation_combo", ["sigmoid_sigmoid", "sigmoid_relu"],
        "activation_combo", ["sigmoid_relu"],
        default="sigmoid_relu"
    )
    if activation_combo == "sigmoid_sigmoid":
        conv_act, head_act = "sigmoid", "sigmoid"
    else:
        conv_act, head_act = "sigmoid", "relu"
    # Stage 1
    k1 = hp.Choice("k1", [5, 7], default=5)
    f1_per_ch = hp.Choice("f1_per_ch", [8, 12], default=8)
    pool1 = hp.Choice("pool1", [2, 3], default=2)
    # Stage 2
    k2 = k1
    f2_per_ch = f1_per_ch // 2
    pool2 = pool1

    # Head
    hidden = hp.Choice("hidden", [256, 512, 768], default=256)
    fc_dropout = hp.Float("fc_dropout", 0.0, 0.4, step=0.2, default=0.2)

    # Optimizer
    lr = hp.Choice("lr", [2e-4, 4e-4, 8e-4], default=2e-4)
    wd = hp.Choice("wd", [1e-5, 3e-5, 1e-4], default=1e-5)

    # Initializers matching activations
    conv_init = _kernel_init_for(conv_act)
    head_init = _kernel_init_for(head_act)

    # ---------------- Model ----------------
    seq_in = L.Input((time_steps, features))

    # Build a two-stage CNN per channel
    per_channel_vecs = []
    for i in range(features):
        ch = ChannelSlice(index=i)(seq_in)  # (B, T, 1)

        # Stage 1: Conv -> BN -> Act -> AvgPool
        x = L.Conv1D(filters=f1_per_ch, kernel_size=k1, padding="same",
                     use_bias=False, kernel_initializer=conv_init)(ch)
        x = L.BatchNormalization(axis=-1)(x)
        x = L.Activation(conv_act)(x)
        x = L.AveragePooling1D(pool_size=pool1, strides=pool1, padding="valid")(x)

        # Stage 2: Conv -> BN -> Act -> AvgPool
        x = L.Conv1D(filters=f2_per_ch, kernel_size=k2, padding="same",
                     use_bias=False, kernel_initializer=conv_init)(x)
        x = L.BatchNormalization(axis=-1)(x)
        x = L.Activation(conv_act)(x)
        x = L.AveragePooling1D(pool_size=pool2, strides=pool2, padding="valid")(x)

        # Flatten this channel's feature maps
        per_channel_vecs.append(L.Flatten()(x))

    # Concatenate all channels’ vectors
    x = L.Concatenate()(per_channel_vecs) if features > 1 else per_channel_vecs[0]

    # Head
    x = L.Dense(hidden, activation=head_act, kernel_initializer=head_init)(x)
    if fc_dropout > 0.0:
        x = L.Dropout(fc_dropout)(x)

    # Output
    out = L.Dense(1, activation="sigmoid", dtype="float32")(x)

    model = M.Model(seq_in, out, name=model_name)

    # Optimizer
    opt = tf.keras.optimizers.AdamW(
        learning_rate=lr, weight_decay=wd, global_clipnorm=clipnorm
    )
    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR",  name="auc_pr"),
            tf.keras.metrics.AUC(curve="ROC", name="auc_roc"),
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
        jit_compile=False  # For compatibility with GPU.
    )
    return model


In [ ]:
tb_log_dir = Path(MODEL_RESULTS_FOLDER) / "logs"
tb_log_dir.mkdir(parents=True, exist_ok=True)
print(f"TensorBoard log directory: {tb_log_dir}")

# Define callbacks
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc_roc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    min_delta=0.002,
    start_from_epoch=3,
    verbose=1
)

reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc_roc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    cooldown=1,
    verbose=1
)

model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(MODEL_RESULTS_FOLDER / "best_model.h5"),
    monitor="val_auc_roc",
    mode="max",
    save_best_only=True,
    verbose=1
)

callbacks = [
    early_stopping_cb,
    reduce_lr_cb,
    model_checkpoint_cb,
]

if USE_TENSORBOARD:
    tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir=tb_log_dir, histogram_freq=1)
    callbacks.append(tensorboard_cb)


## Training of the model

In [ ]:
print("Train set:")
print("X_ts_train_final.shape:", X_ts_train.shape)

In [ ]:
print("Validation set:")
print("X_ts_val.shape:", X_ts_val.shape)

In [ ]:
print("Test set:")
print("X_ts_test.shape:", X_ts_test.shape)

In [ ]:
hyper_model = MetaHyperModel(
    model_name=MODEL_NAME,
    build_model_func=build_model,
    time_steps=X_ts_train.shape[1],
    features=X_ts_train.shape[2],
    clipnorm=0.1,
)
model_loader = ModelLoader(hyper_model, MODEL_RESULTS_FOLDER,
        random_state=RANDOM_STATE,
        max_epochs=70,
        factor=4,
        objective=kt.Objective("val_auc_roc", direction="max"),
        overwrite=True,
    )

print(f"\nCommand to run tensorboard: \npoetry run tensorboard --logdir '{tb_log_dir}'")

tuner = model_loader.get_tuner()
print("\nSearch space summary:")
print("="*50)
tuner.search_space_summary(extended=True)
print("="*50)

In [ ]:
BATCH_SIZE = 128  # Reduced from 256 to 128 due to performance 

if SAVE_MODEL:
    model_history = model_loader.tune_and_train(
        X_ts_train, y_train,
        X_ts_val, y_val,
        class_weight=class_weight_dict,
        epochs=100,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )
    model = model_history.model
    history = model_history.history
    print("Tuning and Training completed!")
else:
    model, history = model_loader.load_keras_model_from_disk()
    print(f"Model loaded from results {model_loader.results_dir}")

In [ ]:
summarize_best_N_models(num_models=5, tuner=tuner, model_summary=False)

## Evaluation of the model:

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 15))

axes[0, 0].plot(history['loss'], label='Training Loss')
axes[0, 0].plot(history['val_loss'], label='Validation Loss')
axes[0, 0].set_title('Model Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

if 'auc_pr' in history and 'val_auc_pr' in history:
    axes[0, 1].plot(history['auc_pr'], label='Training AUC-PR')
    axes[0, 1].plot(history['val_auc_pr'], label='Validation AUC-PR')
    axes[0, 1].set_title('Model AUC-PR')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('AUC-PR')
    axes[0, 1].legend()
else:
    axes[0, 1].set_visible(False)

if 'auc_roc' in history and 'val_auc_roc' in history:
    axes[1, 0].plot(history['auc_roc'], label='Training AUC-ROC')
    axes[1, 0].plot(history['val_auc_roc'], label='Validation AUC-ROC')
    axes[1, 0].set_title('Model AUC-ROC')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('AUC-ROC')
    axes[1, 0].legend()
else:
    axes[1, 0].set_visible(False)

if 'precision' in history and 'val_precision' in history:
    axes[1, 1].plot(history['precision'], label='Training Precision')
    axes[1, 1].plot(history['val_precision'], label='Validation Precision')
    axes[1, 1].set_title('Model Precision')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Precision')
    axes[1, 1].legend()
else:
    axes[1, 1].set_visible(False)

if 'recall' in history and 'val_recall' in history:
    axes[2, 0].plot(history['recall'], label='Training Recall')
    axes[2, 0].plot(history['val_recall'], label='Validation Recall')
    axes[2, 0].set_title('Model Recall')
    axes[2, 0].set_xlabel('Epoch')
    axes[2, 0].set_ylabel('Recall')
    axes[2, 0].legend()
else:
    axes[2, 0].set_visible(False)

if 'accuracy' in history and 'val_accuracy' in history:
    axes[2, 1].plot(history['accuracy'], label='Training Accuracy')
    axes[2, 1].plot(history['val_accuracy'], label='Validation Accuracy')
    axes[2, 1].set_title('Model Accuracy')
    axes[2, 1].set_xlabel('Epoch')
    axes[2, 1].set_ylabel('Accuracy')
    axes[2, 1].legend()
else:
    axes[2, 1].set_visible(False)

plt.tight_layout()
plt.show()

# Save the main plot
plt.savefig(MODEL_RESULTS_FOLDER / f'training_history.png', dpi=300, bbox_inches='tight')

In [ ]:
y_pred_proba_val = model.predict(X_ts_val)
best_threshold, stats = pick_threshold(y_val, y_pred_proba_val.flatten(), method="macro")
print(f"Best threshold: {best_threshold}")
for k, v in stats.items():
    print(f"Best {k}: {v}")

In [ ]:
pred = BilateralSingleInputPredictor(model)
test_summary = model_test_summary(model, X_ts_test, y_test, threshold=best_threshold, predictor=pred)

In [ ]:
# Save the trained model
if SAVE_MODEL:
    model_loader.save_keras_model_to_disk()
    
    # Save scalers for future use
    scalers = {
        'timeseries_scaler': scaler_ts,
    }
    model_loader.save_scalers_to_disk(scalers)

    # Save training results
    results = {
        'test_accuracy': float(test_summary['accuracy']),
        'test_f1': float(test_summary['f1']),
        'tes_avg_precision': float(test_summary['auc_pr']),
        'test_auc_roc': float(test_summary['auc_roc']),
        'test_precision': float(test_summary['precision']),
        'test_recall': float(test_summary['recall']),   
        'test_prevalence': float(test_summary['prevalence']),
        'threshold': float(test_summary['threshold']),
        'model_name': MODEL_NAME,
        'training_params': {
            "epochs": model.history.params['epochs'],
            'epochs_run': len(model.history.epoch),
            'history': model.history.history,
            'batch_size': BATCH_SIZE,
            'learning_rate': float(model.optimizer.learning_rate.numpy()),
            'dropout_rate': next((layer.rate for layer in model.layers if hasattr(layer, 'rate')), "N/A"),
            'class_weights': class_weight_dict
        }
    }
    model_loader.save_results_to_disk(results)

    print(f"Model and results saved!")

## Model Explainability

Let's analyze feature importance using gradient-based saliency maps to understand which features and time points are most important for the model's predictions.


In [ ]:
from core.evaluation import (
    compute_timeseries_saliency,
    plot_timeseries_saliency,
    analyze_sample_saliency,
    get_unilateral_feature_names
)

In [ ]:
print("Left Side Analysis")
print("="*50)
l_feature_names = get_unilateral_feature_names(channels, side="L")
l_saliency = compute_timeseries_saliency(model, X_ts_test, method="vanilla")
l_results = analyze_sample_saliency(model, [X_ts_test], y_test, l_feature_names)

print("Right Side Analysis")
print("="*50)
r_feature_names = get_unilateral_feature_names(channels, side="R")
r_saliency = compute_timeseries_saliency(model, X_ts_test, method="vanilla")
r_results = analyze_sample_saliency(model, [X_ts_test], y_test, r_feature_names)